# 02 · Work Unit Segmentation & Boundary Detection Engine
**Project:** IMBY Enterprise Operations Intelligence & Process Discovery  
**Objective:** Segment continuous user activity streams into discrete, confident business process executions.

---

### Modeling Rationale & Engineering Approach
In operational process discovery, the core technical problem is: **where does one business task end and another begin?**

During early prototyping, deep neural sequence models (such as BiLSTMs) were evaluated for boundary detection. However, deep sequence models suffered from **domain shift** when moving from synthetic training sessions to unlabelled, multi-application production environments—frequently breaking continuous tasks during slow typing pauses or missing transitions altogether. Furthermore, opaque neural predictions provide zero confidence provenance for enterprise audits.

To solve this reliably for enterprise deployment, we designed the **Production Hybrid Segmenter** (`src/segmentation/hybrid_segmenter.py`):
- **Multi-Signal Boundary Detection:** Combines inactivity dwell gaps (e.g. >24s), primary application route shifts, and noise-immune auxiliary window transitions (Word, Excel, Notepad).
- **Hierarchical Confidence Scoring:** Assigns deterministic signal confidence:
  - `0.95`: DOM button action anchor (e.g., `btn-pi-ok`, `btn-la-ok`)
  - `0.85`: Active browser route navigation (`/payroll-items`, `/leave-applications`)
  - `0.75`: Specific operational documentation titles (e.g., `gyomu_itaku_kyuuyo_kitei.docx`)
  - `0.60`: Main window title keywords
  - `0.20`: Unclassified gap fallback (`unknown_or_unclassified`)
- **Decoupled Evaluation:** Boundary temporal F1, segment IoU, and process label accuracy are evaluated and reported independently.

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter
from datetime import datetime

# Robust project root resolution
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "Datasets").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.loader import SessionDataLoader
from src.ingestion.models import Segment
from src.segmentation.hybrid_segmenter import HybridSegmenter
from src.segmentation.classifier import ProcessClassifier
from src.evaluation.evaluator import SegmentationEvaluator

print("Loaded production segmentation and evaluation modules successfully.")

## 1. Inspecting the Hybrid Segmenter Configuration
Let's initialize the segmenter with production-tuned parameters:

In [ ]:
segmenter = HybridSegmenter(
    dwell_gap_seconds=24.0,     # Inactivity threshold between discrete tasks
    min_segment_seconds=6.0,    # Minimum duration for valid business action
    min_segment_events=3,       # Minimum event cluster size
    merge_gap_seconds=6.0      # Merge adjacent segments with identical label
)

print("HybridSegmenter initialized with:")
print(f"  - dwell_gap_seconds  : {segmenter.dwell_gap_seconds}s")
print(f"  - min_segment_seconds: {segmenter.min_segment_seconds}s")
print(f"  - min_segment_events : {segmenter.min_segment_events}")
print(f"  - merge_gap_seconds  : {segmenter.merge_gap_seconds}s")


## 2. Decoupled Benchmark Evaluation on Dataset A
We evaluate the segmenter across the benchmark sessions in Dataset A. We calculate:
1. **Boundary Macro & Micro F1** (tolerance $\pm 10$s)
2. **Mean Segment IoU** (temporal intersection over union)
3. **Process Label Accuracy** (classification accuracy on correctly matched boundaries)

In [ ]:
dataset_a_dir = PROJECT_ROOT / "Datasets" / "dataset_a"
sessions_a = sorted([d for d in dataset_a_dir.iterdir() if d.is_dir()])

evaluator = SegmentationEvaluator(tolerance_seconds=10.0, min_iou=0.25)

# Evaluate across the first 15 benchmark sessions
eval_sessions = sessions_a[:15]
running_f1s = []
running_ious = []
running_label_accs = []

print(f"Evaluating on {len(eval_sessions)} sessions from Dataset A...")
for s in eval_sessions:
    loader = SessionDataLoader(s)
    gt = loader.load_ground_truth()
    if not gt:
        continue
    predicted = segmenter.process_session(s)
    metrics = evaluator.evaluate_session(predicted, gt)
    running_f1s.append(metrics["f1"])
    if metrics["avg_iou"] > 0:
        running_ious.append(metrics["avg_iou"])
    if metrics["matches"] > 0:
        running_label_accs.append(metrics["label_accuracy"])

macro_f1 = sum(running_f1s) / len(running_f1s) if running_f1s else 0.0
macro_iou = sum(running_ious) / len(running_ious) if running_ious else 0.0
macro_label_acc = sum(running_label_accs) / len(running_label_accs) if running_label_accs else 0.0

print("\n--- Decoupled Benchmark Metrics (Dataset A Sample) ---")
print(f"Boundary Macro F1 Score : {macro_f1 * 100:.2f}%")
print(f"Mean Segment IoU        : {macro_iou * 100:.2f}%")
print(f"Process Label Accuracy  : {macro_label_acc * 100:.2f}%")

## 3. Production Segmentation on Dataset B
Now we apply the segmenter to all 15 unlabelled production sessions in `Datasets/dataset_b`.

In [ ]:
dataset_b_dir = PROJECT_ROOT / "Datasets" / "dataset_b"
recovered_segments = segmenter.process_all_sessions(dataset_b_dir)

print(f"Total Recovered Segments across Dataset B: {len(recovered_segments)}")

conf_scores = [s.confidence for s in recovered_segments]
mean_conf = sum(conf_scores) / len(conf_scores) if conf_scores else 0.0
print(f"Mean Signal Confidence Score             : {mean_conf:.2f}")

label_counts = Counter(s.label for s in recovered_segments)
print("\nProcess Distribution across Recovered Segments:")
for label, count in label_counts.most_common():
    share = (count / len(recovered_segments)) * 100.0
    print(f"  {label:<32}: {count:>3} segments ({share:>5.1f}%)")

## 4. Visual Inspection of Recovered Segments & Evidence Chains
Let's inspect the first 5 recovered segments from the first production session, verifying their timestamps, confidence score, detection method, and recorded evidence.

In [ ]:
print(f"{'Session':<20} {'Label':<30} {'Dur(s)':>6} {'Conf':>5} {'Method':<18} {'Evidence'}")
print("-" * 95)
for s in recovered_segments[:8]:
    sess_short = s.session_id[:18]
    dur = f"{s.duration_seconds:.1f}"
    conf = f"{s.confidence:.2f}"
    ev_sample = ", ".join(s.evidence[:2]) if s.evidence else "none"
    ev_safe = ev_sample.encode("ascii", errors="backslashreplace").decode("ascii")
    print(f"{sess_short:<20} {s.label:<30} {dur:>6} {conf:>5} {s.detection_method:<18} {ev_safe}")


## 5. Deliverable Export
We serialize the recovered segments to `deliverables/segments.jsonl` matching the required schema.

In [ ]:
out_path = PROJECT_ROOT / "deliverables" / "segments.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    for seg in recovered_segments:
        f.write(json.dumps(seg.to_dict(), ensure_ascii=False) + "\n")

print(f"Successfully written {len(recovered_segments)} segments to {out_path}.")